# 03b - Accommodation (Airbnb / short-term rental)

Adds the third price layer: a per-location **entire-home nightly rate** so the app
can estimate what a group pays to sleep, alongside flights and the lifestyle
basket. Built to match what travellers *actually* pay on Airbnb, not the
advertised base rate.

## Source + method (same shape as 03_costs)

- **Anchors -> [Inside Airbnb](https://insideairbnb.com/get-the-data/)** (free,
  CC-BY 4.0, listing-level). For each covered city we take the **median** nightly
  price of an **entire home/apt** sized to a group, captured June 2026. ~30
  European cities are covered directly (incl. Brussels, Antwerp, Ghent).
- **The long tail** - countries without a covered city are scaled from the
  Belgium baseline by the **Eurostat 2024 restaurants & hotels PLI** (the closest
  official proxy for lodging price levels).
- **Validation** - the anchored countries are re-predicted from their PLI and the
  mean error is reported, so the long-tail scaling is measurable.

## The five accuracy fixes (vs a naive average)

1. **Median, not mean** - Airbnb prices are right-skewed; the median entire-home
   rate is the honest central estimate. (Baked into the anchors.)
2. **Real fees** - Inside Airbnb's `price` excludes the cleaning fee and Airbnb's
   ~14% guest service fee. We model a per-booking cleaning fee (`cleaning_per_person_eur`)
   and a `service_fee_pct`, applied at runtime so the cleaning fee amortises over
   the trip length.
3. **Capacity-matched, per person** - we filter to entire homes and divide the
   nightly by the listing's capacity, so the figure is honest per head for a group.
4. **Seasonality** - the anchor is an annual median; a monthly multiplier lifts
   summer (the app's May-Aug window runs well above average).
5. **Length-of-stay discount** - stays of >= 7 nights get the common weekly discount.

Fixes 2, 4 and 5 depend on the user's dates/length, so they live in
`runtime_pricing.js` and `meta.accommodation_model`; this notebook ships the
anchors plus those model parameters. Writes `cache/accommodation.json`.

Refresh yearly: replace the curated anchors with real medians using the
`compute_anchor_from_csv` helper below on a fresh Inside Airbnb download.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

CACHE_DIR = Path("cache")
cfg    = json.loads((CACHE_DIR / "config.json").read_text(encoding="utf-8"))
master = json.loads((CACHE_DIR / "destinations_master.json").read_text(encoding="utf-8"))
dests  = master["destinations"]

BASELINE = "BE"

# --- Accommodation model (applied at runtime in runtime_pricing.js) ---
# Turns the stored annual-median nightly into what a traveller actually pays.
CLEAN_FRAC = 0.5          # cleaning fee per booking ~= half of one whole-home night
SERVICE_FEE_PCT = 14.0    # Airbnb guest service fee (~14%)
WEEKLY_DISCOUNT_PCT = 8.0 # typical discount for stays >= 7 nights
MIN_NIGHTS_FOR_WEEKLY = 7

# Monthly multiplier on the nightly rate (annual median = ~1.0). Summer peaks;
# matches AirDNA European-review seasonality (leisure ADR ~20-40% over average).
SEASONALITY = {1:0.82, 2:0.82, 3:0.90, 4:0.98, 5:1.08, 6:1.22,
               7:1.35, 8:1.35, 9:1.15, 10:1.00, 11:0.85, 12:0.92}

print(f"Destinations: {len(dests)}  |  countries: {len(set(d['iso2'] for d in dests))}")
print(f"Model: cleaning={CLEAN_FRAC} night, service={SERVICE_FEE_PCT}%, "
      f"weekly -{WEEKLY_DISCOUNT_PCT}% (>= {MIN_NIGHTS_FOR_WEEKLY}n)")

Destinations: 450  |  countries: 42
Model: cleaning=0.5 night, service=14.0%, weekly -8.0% (>= 7n)


## 1. Inside Airbnb anchors

Median **entire home/apt** nightly price (EUR, base only - no fees) and the
listing's typical capacity, captured from Inside Airbnb city files (June 2026).
City overrides sit on top of country-level anchors, exactly like the cost basket.

In [2]:
# city -> (median entire-home nightly EUR, typical capacity). Inside Airbnb medians.
ACCOM_CITY = {
    "Amsterdam": (185, 4), "Antwerp": (115, 4), "Athens": (80, 4),
    "Barcelona": (130, 4), "Berlin": (115, 4), "Bologna": (120, 4),
    "Bordeaux": (115, 4), "Bristol": (130, 4), "Brussels": (110, 4),
    "Budapest": (90, 4),   "Copenhagen": (165, 4), "Crete": (100, 4),
    "Dublin": (165, 4),    "Edinburgh": (150, 4),  "Florence": (145, 4),
    "Geneva": (175, 4),    "Ghent": (115, 4),      "Girona": (110, 4),
    "Manchester": (120, 4),"Istanbul": (55, 4),    "Lisbon": (125, 4),
    "London": (175, 4),    "Lyon": (115, 4),       "Madrid": (120, 4),
    "Malaga": (110, 4),    "Mallorca": (145, 4),   "Milan": (135, 4),
    "Munich": (140, 4),    "Naples": (95, 4),      "Paris": (160, 4),
    "Rome": (130, 4),      "Vienna": (115, 4),
}

# Country-level anchors where Inside Airbnb covers the major cities directly.
# (median entire-home nightly EUR, typical capacity)
ACCOM_COUNTRY = {
    "BE": (112, 4), "NL": (175, 4), "FR": (130, 4), "ES": (122, 4),
    "IT": (125, 4), "GR": (90, 4),  "PT": (125, 4), "DE": (125, 4),
    "AT": (115, 4), "HU": (90, 4),  "GB": (150, 4), "IE": (165, 4),
    "DK": (165, 4), "CH": (175, 4), "TR": (55, 4),
}

# --- Eurostat 2024 restaurants & hotels PLI (EU27_2020=100) - long-tail scaling.
# Closest official proxy for lodging price levels. Same source as 03_costs.
PLI_REST = {
    "AT":117,"BE":118,"BG":62,"HR":85,"CY":93,"CZ":85,"DK":165,"EE":98,"FI":138,
    "FR":109,"DE":116,"GR":88,"HU":72,"IE":150,"IT":102,"LV":88,"LT":79,"LU":132,
    "MT":100,"NL":123,"PL":64,"PT":85,"RO":58,"SK":85,"SI":90,"ES":96,"SE":116,
    "CH":180,"IS":175,"NO":145,"GB":134,"AL":55,"BA":63,"ME":68,"MK":48,"RS":68,
}
NEIGHBOR_FALLBACK = {
    "AD":["ES","FR"], "MC":["FR"], "SM":["IT"], "LI":["CH","AT"], "XK":["MK","RS"],
    "MD":["RO"], "VA":["IT"], "UA":["RO","PL"], "BY":["PL","LT"],
    "FO":["DK"],  # Faroe Islands: Denmark proxy
}

def pli_for(iso):
    """restaurants & hotels PLI for a country, with neighbour fallback. None if unknown."""
    if iso in PLI_REST:
        return PLI_REST[iso]
    vals = [PLI_REST[n] for n in NEIGHBOR_FALLBACK.get(iso, []) if n in PLI_REST]
    return sum(vals) / len(vals) if vals else None

def rec(night, cap, source):
    """Build the stored accommodation record from a whole-home nightly + capacity."""
    ppn = round(night / cap, 2)
    return {
        "per_person_night_eur":    ppn,                       # entire-home base, per head
        "cleaning_per_person_eur": round(CLEAN_FRAC * ppn, 2),# per booking, per head
        "entire_home_night_eur":   round(night, 2),           # headline (whole home)
        "typical_capacity":        cap,
        "source":                  source,
    }

BE_NIGHT, BE_CAP = ACCOM_COUNTRY[BASELINE]
BE_PPN  = BE_NIGHT / BE_CAP
BE_REST = PLI_REST[BASELINE]
print(f"Baseline {BASELINE}: EUR {BE_NIGHT}/night entire home (cap {BE_CAP}) = EUR {BE_PPN:.2f} pp/night")
print(f"Anchors: {len(ACCOM_COUNTRY)} countries direct, {len(ACCOM_CITY)} cities")

Baseline BE: EUR 112/night entire home (cap 4) = EUR 28.00 pp/night
Anchors: 15 countries direct, 32 cities


## Refresh helper - real medians from an Inside Airbnb download

Run this on a fresh `listings.csv[.gz]` to replace a curated anchor with the real
median. It applies fixes 1 and 3 (median entire-home, capacity-matched, outliers
trimmed). Modern Inside Airbnb dumps no longer ship a `cleaning_fee` column, which
is exactly why the cleaning fee is modelled as a fraction of a night above.

In [3]:
def compute_anchor_from_csv(path, min_capacity=2, room_type="Entire home/apt"):
    """Median entire-home nightly + capacity from an Inside Airbnb listings file.
    Returns a dict ready to drop into ACCOM_CITY / ACCOM_COUNTRY. Not run at
    build time (needs a download); here as the documented refresh recipe."""
    import pandas as pd
    df = pd.read_csv(path)
    if df["price"].dtype == object:                       # detailed dumps: "$1,234.00"
        df["price"] = df["price"].replace(r"[\$,]", "", regex=True).astype(float)
    df = df[(df["room_type"] == room_type) & (df["accommodates"] >= min_capacity)]
    lo, hi = df["price"].quantile([0.01, 0.99])           # trim outliers (fix 1)
    df = df[(df["price"] >= lo) & (df["price"] <= hi)]
    return {
        "entire_home_night": round(float(df["price"].median()), 2),
        "typical_capacity":  int(df["accommodates"].median()),
        "n_listings":        int(len(df)),
    }

# Example (uncomment with a real file):
# print(compute_anchor_from_csv("inside_airbnb/brussels_listings.csv.gz"))

## 2. Build the per-country anchor

Direct Inside Airbnb anchor where we have one; otherwise scale the Belgium
baseline by the country's restaurants & hotels PLI.

In [4]:
def country_accom(iso):
    if iso in ACCOM_COUNTRY:
        night, cap = ACCOM_COUNTRY[iso]
        return rec(night, cap, "inside_airbnb_country")
    rest = pli_for(iso)
    if rest is not None:
        return rec(BE_NIGHT * (rest / BE_REST), BE_CAP, "airbnb_pli_scaled")
    return None

dest_isos = sorted({d["iso2"] for d in dests})
countries, missing = {}, []
for iso in dest_isos:
    a = country_accom(iso)
    countries[iso] = a if a else missing.append(iso)
countries = {k: v for k, v in countries.items() if v}

from collections import Counter
src = Counter(a["source"] for a in countries.values())
print(f"Country anchors: {len(countries)}  sources={dict(src)}")
if missing:
    print(f"  no data (app falls back to baseline): {missing}")
print("\nSample (EUR):")
for iso in ["BE","ES","IT","FR","GR","PT","PL","HR","CZ","RO","CH","NO"]:
    if iso in countries:
        a = countries[iso]
        print(f"  {iso}: EUR {a['entire_home_night_eur']:>6.0f}/night whole home "
              f"(cap {a['typical_capacity']}) = EUR {a['per_person_night_eur']:>5.2f} pp/night "
              f"+ EUR {a['cleaning_per_person_eur']:.2f} clean  ({a['source']})")

Country anchors: 42  sources={'airbnb_pli_scaled': 28, 'inside_airbnb_country': 14}

Sample (EUR):
  BE: EUR    112/night whole home (cap 4) = EUR 28.00 pp/night + EUR 14.00 clean  (inside_airbnb_country)
  ES: EUR    122/night whole home (cap 4) = EUR 30.50 pp/night + EUR 15.25 clean  (inside_airbnb_country)
  IT: EUR    125/night whole home (cap 4) = EUR 31.25 pp/night + EUR 15.62 clean  (inside_airbnb_country)
  FR: EUR    130/night whole home (cap 4) = EUR 32.50 pp/night + EUR 16.25 clean  (inside_airbnb_country)
  GR: EUR     90/night whole home (cap 4) = EUR 22.50 pp/night + EUR 11.25 clean  (inside_airbnb_country)
  PT: EUR    125/night whole home (cap 4) = EUR 31.25 pp/night + EUR 15.62 clean  (inside_airbnb_country)
  PL: EUR     61/night whole home (cap 4) = EUR 15.19 pp/night + EUR 7.59 clean  (airbnb_pli_scaled)
  HR: EUR     81/night whole home (cap 4) = EUR 20.17 pp/night + EUR 10.09 clean  (airbnb_pli_scaled)
  CZ: EUR     81/night whole home (cap 4) = EUR 20.17 pp/night

## 3. City overrides

The Inside Airbnb cities override their country (capitals/hubs run above the
national average).

In [5]:
cities = {city: rec(night, cap, "inside_airbnb_city")
          for city, (night, cap) in ACCOM_CITY.items()}

print(f"City overrides: {len(cities)}")
for city, a in sorted(cities.items()):
    print(f"  {city:<12} EUR {a['entire_home_night_eur']:>5.0f}/night "
          f"= EUR {a['per_person_night_eur']:>5.2f} pp/night")

City overrides: 32
  Amsterdam    EUR   185/night = EUR 46.25 pp/night
  Antwerp      EUR   115/night = EUR 28.75 pp/night
  Athens       EUR    80/night = EUR 20.00 pp/night
  Barcelona    EUR   130/night = EUR 32.50 pp/night
  Berlin       EUR   115/night = EUR 28.75 pp/night
  Bologna      EUR   120/night = EUR 30.00 pp/night
  Bordeaux     EUR   115/night = EUR 28.75 pp/night
  Bristol      EUR   130/night = EUR 32.50 pp/night
  Brussels     EUR   110/night = EUR 27.50 pp/night
  Budapest     EUR    90/night = EUR 22.50 pp/night
  Copenhagen   EUR   165/night = EUR 41.25 pp/night
  Crete        EUR   100/night = EUR 25.00 pp/night
  Dublin       EUR   165/night = EUR 41.25 pp/night
  Edinburgh    EUR   150/night = EUR 37.50 pp/night
  Florence     EUR   145/night = EUR 36.25 pp/night
  Geneva       EUR   175/night = EUR 43.75 pp/night
  Ghent        EUR   115/night = EUR 28.75 pp/night
  Girona       EUR   110/night = EUR 27.50 pp/night
  Istanbul     EUR    55/night = EUR 13.75 pp

## 4. Validate the PLI scaling against the anchors

Re-predict each directly-anchored country's per-person nightly from the Belgium
baseline x restaurants & hotels PLI, and compare to the real anchor. Higher error
than the dining basket is expected - Airbnb in tourist hubs outruns the national
price level - which is exactly why covered cities/countries keep real anchors and
only the long tail is scaled.

In [6]:
rows, errs = [], []
for iso, (night, cap) in ACCOM_COUNTRY.items():
    p = pli_for(iso)
    if iso == BASELINE or p is None:   # p is None: no Eurostat PLI (e.g. non-EEA)
        continue
    actual = night / cap
    est = BE_PPN * (p / BE_REST)
    err = 100 * (est - actual) / actual
    errs.append(abs(err))
    rows.append((iso, actual, est, err))

print(f"{'ISO':>4} {'anchor_pp':>9} {'pli_est':>8} {'err%':>7}")
for iso, actual, est, err in sorted(rows):
    print(f"{iso:>4} {actual:>9.2f} {est:>8.2f} {err:>+7.1f}")

overall = sum(errs) / len(errs)
print(f"\nMean absolute error (long-tail PLI scaling): {overall:.1f}%  over {len(errs)} countries")
print("Covered cities/countries use real Inside Airbnb medians, so this error only")
print("applies to the smaller long-tail markets scaled from Belgium.")
validation = {
    "method": "Belgium baseline x Eurostat restaurants&hotels PLI, validated vs Inside Airbnb anchors",
    "overall_mae_pct": round(overall, 1),
    "n_checks": len(errs),
}

 ISO anchor_pp  pli_est    err%
  AT     28.75    27.76    -3.4
  CH     43.75    42.71    -2.4
  DE     31.25    27.53   -11.9
  DK     41.25    39.15    -5.1
  ES     30.50    22.78   -25.3
  FR     32.50    25.86   -20.4
  GB     37.50    31.80   -15.2
  GR     22.50    20.88    -7.2
  HU     22.50    17.08   -24.1
  IE     41.25    35.59   -13.7
  IT     31.25    24.20   -22.5
  NL     43.75    29.19   -33.3
  PT     31.25    20.17   -35.5

Mean absolute error (long-tail PLI scaling): 16.9%  over 13 countries
Covered cities/countries use real Inside Airbnb medians, so this error only
applies to the smaller long-tail markets scaled from Belgium.


## 5. Save

In [7]:
OUT = CACHE_DIR / "accommodation.json"
model = {
    "service_fee_pct":        SERVICE_FEE_PCT,
    "cleaning_fee_frac_of_night": CLEAN_FRAC,
    "weekly_discount_pct":    WEEKLY_DISCOUNT_PCT,
    "min_nights_for_weekly":  MIN_NIGHTS_FOR_WEEKLY,
    "seasonality":            {str(k): v for k, v in SEASONALITY.items()},
    "assumptions": ("entire home/apt; stored value is the annual median base nightly "
                    "per person (whole-home median / capacity). Runtime adds seasonality, "
                    "a weekly discount, an amortised cleaning fee and the service fee."),
}
payload = {
    "meta": {
        "schema_version": cfg["schema_version"],
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "currency":       "EUR",
        "baseline":       BASELINE,
        "model":          model,
        "sources": [
            "Inside Airbnb listing-level data, CC-BY 4.0, captured June 2026 (anchors)",
            "Eurostat 2024 restaurants & hotels Price Level Indices (long-tail scaling)",
            "AirDNA European market reviews 2025 (seasonality calibration)",
        ],
        "validation": validation,
    },
    "countries": countries,
    "cities":    cities,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {OUT}  ({OUT.stat().st_size/1024:.1f} KB)")
print(f"  {len(countries)} countries, {len(cities)} cities")
print(f"  long-tail PLI scaling error: {validation['overall_mae_pct']}%")

Wrote cache\accommodation.json  (16.3 KB)
  42 countries, 32 cities
  long-tail PLI scaling error: 16.9%
